# 📊 06 — Evaluasi Model Hasil Abliterasi

**Tujuan:** Hitung RR dan ASR heuristik model yang sudah di-abliterasi, bandingkan dengan baseline.

**Lokasi:** VS Code (forward pass ringan)

---
**Output:** `results/generations_log/abliterated_harmful.csv` + append ke `results/metrics.csv`

Saved results are in `results/reference/`. Your results may differ with other inputs or settings. See `docs/results.md` for details.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "src" / "config.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open the notebook from inside the project directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.config import ABLITERATED_HF_DIR, N_INST_TRAIN, N_INST_TEST, DEVICE, LOGS_DIR, RESULTS_DIR, SEED
from src.data_loader import get_harmful_instructions, get_harmless_instructions, create_holdout_set, get_train_test_split
from src.generation_utils import get_generations_hf
from src.metrics import compute_refusal_rate, compute_asr, classify_refusal

torch.manual_seed(SEED)
print(f"Device: {DEVICE}")

In [ ]:
# ─── Load model abliterasi ───
model = AutoModelForCausalLM.from_pretrained(
    str(ABLITERATED_HF_DIR),
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
tokenizer = AutoTokenizer.from_pretrained(str(ABLITERATED_HF_DIR))
tokenizer.pad_token = tokenizer.eos_token
print(f"Model loaded from: {ABLITERATED_HF_DIR}")

In [ ]:
# ─── Load test data ───
harmful = get_harmful_instructions()
harmless = get_harmless_instructions()
holdout_harmful, holdout_harmless = create_holdout_set(harmful, harmless)
_, _, harmful_test, harmless_test = get_train_test_split(harmful, harmless, holdout_harmful, holdout_harmless, n_train=N_INST_TRAIN, n_test=N_INST_TEST)
print(f"Test: {len(harmful_test)} harmful + {len(harmless_test)} harmless")

In [ ]:
# ─── Generate ───
print("Generating with abliterated model...")
ablit_harmful = get_generations_hf(model, tokenizer, harmful_test)
ablit_harmless = get_generations_hf(model, tokenizer, harmless_test)

# Tampilkan contoh
for i in range(min(3, len(ablit_harmful))):
    refused = classify_refusal(ablit_harmful[i])
    print(f"\n--- Prompt {i} (Refused: {refused}) ---")
    print(f"Q: {harmful_test[i]}")
    print(f"A: {ablit_harmful[i][:200]}")

In [ ]:
# ─── Metrik ───
ablit_rr = compute_refusal_rate(ablit_harmful)
ablit_asr = compute_asr(ablit_harmful, harmful_test)

print(f"ABLITERATED METRICS")
print(f"Refusal Rate (RR):  {ablit_rr:.2%}")
print(f"ASR heuristic: {ablit_asr:.2%}")
print(f"Sample size: {len(ablit_harmful)}")

# Bandingkan dengan baseline (load dari metrics.csv)
baseline_metrics = pd.read_csv(RESULTS_DIR / 'metrics.csv')
baseline_rr = baseline_metrics[baseline_metrics['condition']=='baseline']['refusal_rate'].values[0]
print(f"\nPerubahan RR: {baseline_rr:.2%} → {ablit_rr:.2%} ({ablit_rr - baseline_rr:+.2%})")

In [ ]:
# ─── Simpan ───
# Log generasi
df_gen = pd.DataFrame({
    'prompt': harmful_test,
    'generation': ablit_harmful,
    'refused': [classify_refusal(g) for g in ablit_harmful],
    'condition': 'abliterated'
})
df_gen.to_csv(LOGS_DIR / 'abliterated_harmful.csv', index=False)
print(f"Saved: {LOGS_DIR / 'abliterated_harmful.csv'}")

# Append ke metrics.csv
df_metrics = pd.DataFrame([{
    'condition': 'abliterated',
    'refusal_rate': ablit_rr,
    'asr_heuristic': ablit_asr,
    'n_samples': len(ablit_harmful),
    'model': 'Llama-3.2-1B-Instruct / abliterated HF',
}])
df_metrics.to_csv(RESULTS_DIR / 'metrics.csv', index=False, mode='a', header=False)
print(f"Appended to: {RESULTS_DIR / 'metrics.csv'}")

---
**Next:** Lanjut ke `07_convert_gguf.ipynb` untuk deployment test.